# Sample and label frog calls

Sample random 10-second segments from your recordings, label the frog calls in them, and export the results to send to the project lead. The labeling rules are in `PROTOCOL.md`.

**You need:** your recordings, in any folder.

**Edit:** the cell in step 1 marked **(edit)**. Optionally, you can edit `ADD_ROUND` in step 3 if you want to add more segments.

**Run** in JupyterLab with **Run All**. Rerunning is safe: it reuses the segments you already have and only adds more when you ask in step 3.

## 1. Input (edit)

Describe your recordings below. The next cell lists them and picks up to 25 random 10-second segments per site: 5 nights spread across your recordings, and 5 segments spread through each night. Everything it makes (clips, segments, labels, settings) is saved in `exports/<SOURCE>/`; your audio folder is only read. To start a separate dataset, use a new `SOURCE`.

Nights come from the date and time in file names. `HOURS` and `MONTHS` only apply to files with a time in their name; a file without one counts as its own night.

`PATH_PATTERN`: where the site name is in each file path, relative to `AUDIO_DIR`:
- `{site}` the site name (required)
- `{_}` any text to skip/ignore
- `{name}` any text to keep as a column, e.g. `{deployment}` or `{recorder}`

| Your files look like | `PATH_PATTERN` |
|---|---|
| `POND3/20260412_213000.wav` | `"{site}/{_}"` |
| `2026-spring/POND3/SD01/file.wav` | `"{deployment}/{site}/{recorder}/{_}"` |
| `POND3_20260412_213000.wav` (no folders; the site name can't contain `_`) | `"{site}_{_}"` |

In [46]:
# EDIT THIS CELL FOR YOUR PROJECT

SOURCE = "test5"                    # short name for your dataset: lowercase letters, digits, hyphens, e.g. "ucb-ponds"
CONTRIBUTOR = "test"               # data contributor name, e.g. "John Doe"
ANNOTATOR = "AV"               # annotator's initials; labels save to exports/<SOURCE>/labels/
AUDIO_DIR = "/Users/amy/Desktop/audio/samples_devicestatic/"       # folder with recordings; use / even on Windows, e.g. "D:/recordings"
PATH_PATTERN = "{site}_{_}_{date}_{time}_{_}"    # where the site name is in each file's path (see the examples above)
TIME_FORMAT = "%Y%m%d_%H%M%S"  # how date and time appear in file names; None if they don't
FILE_TIMEZONE = "local"          # time zone the recorder wrote (AudioMoth: usually UTC); "local" if local time
LOCAL_TIMEZONE = "America/Los_Angeles"             # your time zone if UTC is used, in California use "America/Los_Angeles"
SITES = None                   # sites to sample, e.g. ["POND3", "POND7"]; None = all sites in AUDIO_DIR
HOURS = [20,21,22,23,0,1,2,3]  # local hours to sample, e.g. [18, 19, 20, 21, 22, 23, 0, 1, 2, 3, 4, 5] = 6 pm to 6 am; None = all
MONTHS = None                  # months to sample, e.g. [3, 4, 5] = March to May; None = all

In [47]:
import importlib
import os
from pathlib import Path

# Work from the repo folder, so every path here and in annotator_config/ starts there.
if Path.cwd().name == "notebooks":
    os.chdir("..")
import functions as fx
importlib.reload(fx)  # use the latest functions.py without restarting the kernel


cfg = fx.load_config(source=SOURCE, contributor=CONTRIBUTOR, audio_dir=AUDIO_DIR, path_pattern=PATH_PATTERN,
                     time_format=TIME_FORMAT, file_timezone=FILE_TIMEZONE, local_timezone=LOCAL_TIMEZONE,
                     sites=SITES, hours=HOURS, months=MONTHS)
fx.sample(cfg)

159 recordings at 6 sites -> exports/test5/inventory.csv
       files  hours with_time pattern_ok
site                                    
COWO4     32    0.3      100%       100%
DOWN1     32    0.3      100%       100%
RW19      32    0.3      100%       100%
STV2      15    0.1      100%       100%
USR1      16    0.1      100%       100%
USRO2     32    0.3      100%       100%
Fewer segments than asked:
  COWO4: 16 of 25 (no room left in its recordings)
  DOWN1: 16 of 25 (no room left in its recordings)
  RW19: 16 of 25 (no room left in its recordings)
  STV2: 15 of 25 (no room left in its recordings)
  USR1: 16 of 25 (no room left in its recordings)
  USRO2: 16 of 25 (no room left in its recordings)
Clips: 95 written, 0 already there
Round 1: added 95 segments. 95 segments in exports/test5/segments.csv.
       segments  recordings  most_from_one_recording
site                                                
COWO4        16          16                        1
DOWN1        16     

## 2. Annotate

Label every segment, following the instructions panel in the tool (full rules in `PROTOCOL.md`). Answers save to `exports/<SOURCE>/labels/frog_labels_<initials>.csv` as you go, so you can stop and come back later.

In [48]:
ann = fx.annotate(cfg, ANNOTATOR)

validation failed: ['"data_index_column" is required when a form is configured — set it to the column that uniquely identifies each row in the input data']
/Users/amy/Desktop/bioacoustic-review/functions.py:562: UserWarning: Config error: "data_index_column" is required when a form is configured — set it to the column that uniquely identifies each row in the input data
  ann.open()


<IPython.core.display.Javascript object>

## 3. Resample (OPTIONAL)

Do you not find any frogs? Need to keep sampling? Set `ADD_ROUND = True`, run the two cells below, then rerun Annotate above in step 2. Samples up to 25 more segments per site, from nights not sampled yet; it never repeats earlier segments and stops once a site reaches the maximum (75). Set `ADD_ROUND` back to `False` afterwards.

In [ ]:
ADD_ROUND = True  # True to sample another round; set back to False afterwards

In [ ]:
fx.add_round(cfg, ADD_ROUND)

## 4. Export

Checks your labels and any missing information, such as a **Yes** without a window: find the segment by its `id` in the tool's table, fix it and submit again. When there's nothing to fix, it zips `exports/<SOURCE>/` (clips, segments, labels and settings) into `exports/<SOURCE>_<date>.zip`. Share that file with the project lead. You can export again later if you add more data; a new zip replaces the old one without losing previous data.

In [ ]:
zip_path = fx.export_source(cfg)